# Exploration du dataset CWRU — Détection d'anomalies vibratoires

**Objectif :** Explorer le dataset CWRU (Case Western Reserve University Bearing Dataset),
comprendre la structure des signaux d'accélération, et extraire des premières features
statistiques permettant de distinguer les quatre états de roulement.

**Dataset :** Signaux vibratoires acquis à **12 000 Hz** sur un banc d'essai moteur.
Quatre états sont représentés :

| Préfixe fichier | Catégorie | Description |
|---|---|---|
| `Normal_*.mat` | Normal | Roulement sain (baseline de référence) |
| `B*.mat` | Ball | Défaut sur la bille roulante |
| `IR*.mat` | IR — Inner Race | Défaut sur la bague intérieure |
| `OR*.mat` | OR — Outer Race | Défaut sur la bague extérieure |

Chaque fichier `.mat` contient un signal Drive End (capteur côté charge, noté `_DE_time`)
qui sera le signal d'entrée de notre autoencoder.

---
**Plan du notebook :**
1. Imports et configuration
2. Inventaire du dataset
3. Lecture et structure d'un fichier `.mat`
4. Comparaison visuelle Normal vs Défauts
5. Features statistiques temporelles
6. Visualisation comparative des features
7. Conclusion et perspectives

## Section 1 — Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io
import scipy.stats
from pathlib import Path

# Configuration matplotlib
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Fréquence d'échantillonnage CWRU (Hz)
SAMPLING_RATE = 12_000

# Chemin vers les données brutes
# Le notebook est dans notebooks/, data/ est à la racine du projet
DATA_RAW = Path("..") / "data" / "raw"

# --- Vérification du dossier ---
if not DATA_RAW.exists():
    raise FileNotFoundError(
        f"Dossier introuvable : {DATA_RAW.resolve()}\n"
        "→ Créez le dossier data/raw/ à la racine et placez-y les fichiers .mat CWRU."
    )

fichiers_mat = sorted(DATA_RAW.glob("*.mat"))
if not fichiers_mat:
    raise FileNotFoundError(
        f"Aucun fichier .mat trouvé dans {DATA_RAW.resolve()}\n"
        "→ Vérifiez que les fichiers CWRU ont bien été copiés dans data/raw/."
    )

print(f"✓ Dossier data/raw/ trouvé : {DATA_RAW.resolve()}")
print(f"✓ {len(fichiers_mat)} fichiers .mat détectés")

## Section 2 — Inventaire du dataset

In [ ]:
def categoriser_fichier(nom_stem: str) -> str:
    """Détermine la catégorie d'un fichier CWRU à partir du préfixe de son nom."""
    nom = nom_stem.upper()
    if nom.startswith("NORMAL"):
        return "Normal"
    if nom.startswith("IR"):
        return "IR"
    if nom.startswith("OR"):
        return "OR"
    if nom.startswith("B"):
        return "Ball"
    return "Inconnu"


# Construction du DataFrame d'inventaire complet
inventaire = [
    {
        "fichier": f.name,
        "categorie": categoriser_fichier(f.stem),
        "taille_ko": round(f.stat().st_size / 1024, 1),
    }
    for f in fichiers_mat
]
df_inventaire = pd.DataFrame(inventaire)

# Récapitulatif par catégorie
df_recap = (
    df_inventaire
    .groupby("categorie", sort=False)
    .agg(
        nb_fichiers=("fichier", "count"),
        taille_totale_mo=("taille_ko", lambda x: round(x.sum() / 1024, 2)),
        exemples=("fichier", lambda x: ", ".join(list(x)[:2])),
    )
    .reset_index()
)

print("=== Inventaire du dataset CWRU ===\n")
df_recap

In [ ]:
# Bar plot de la répartition par catégorie
COULEURS = {"Normal": "#4CAF50", "Ball": "#FF9800", "IR": "#F44336", "OR": "#2196F3", "Inconnu": "#9E9E9E"}

categories = df_recap["categorie"].tolist()
nb_fichiers = df_recap["nb_fichiers"].tolist()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    categories,
    nb_fichiers,
    color=[COULEURS.get(c, "#9E9E9E") for c in categories],
    edgecolor="white",
    linewidth=1.2,
    width=0.55,
)
for bar, val in zip(bars, nb_fichiers):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        str(val),
        ha="center", fontweight="bold", fontsize=12,
    )

ax.set_title("Répartition des fichiers .mat par catégorie de défaut", pad=12)
ax.set_xlabel("Catégorie")
ax.set_ylabel("Nombre de fichiers")
ax.set_ylim(0, max(nb_fichiers) + 2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Section 3 — Lecture d'un fichier `.mat`

In [ ]:
# Ouverture d'un fichier Normal pour inspecter la structure
fichier_exemple = next(DATA_RAW.glob("Normal_*.mat"))
print(f"Fichier ouvert : {fichier_exemple.name}\n")

mat_data = scipy.io.loadmat(str(fichier_exemple))

print("=== Toutes les clés du dictionnaire .mat ===")
print("(les clés '__header__', '__version__', '__globals__' sont des métadonnées scipy)\n")
for cle, valeur in mat_data.items():
    if cle.startswith("__"):
        # Métadonnées internes scipy — ignorées dans l'analyse
        continue
    shape = getattr(valeur, "shape", type(valeur).__name__)
    dtype = getattr(valeur, "dtype", "")
    print(f"  {cle!r:30s}  shape={shape}   dtype={dtype}")

In [ ]:
# Extraction du signal Drive End (clé se terminant par '_DE_time')
# Le préfixe numérique varie selon le fichier (X097_, X105_, X118_, etc.)
de_keys = [k for k in mat_data.keys() if k.endswith("_DE_time")]

if not de_keys:
    raise KeyError("Aucune clé '_DE_time' trouvée dans ce fichier .mat")

cle_de = de_keys[0]
print(f"Clé Drive End identifiée : {cle_de!r}\n")

# .flatten() : passage de (N, 1) à (N,) — array 1D
signal_exemple = mat_data[cle_de].flatten()

duree_totale_s = len(signal_exemple) / SAMPLING_RATE

print(f"{'Shape':<18}: {signal_exemple.shape}")
print(f"{'dtype':<18}: {signal_exemple.dtype}")
print(f"{'Durée totale':<18}: {duree_totale_s:.2f} s  ({len(signal_exemple):,} points à {SAMPLING_RATE} Hz)")
print(f"{'Min':<18}: {signal_exemple.min():.6f}")
print(f"{'Max':<18}: {signal_exemple.max():.6f}")
print(f"{'Moyenne':<18}: {signal_exemple.mean():.6f}")
print(f"{'Écart-type':<18}: {signal_exemple.std():.6f}")

### Observations sur la structure

- Chaque fichier `.mat` contient **3 variables utiles** : le signal Drive End (`_DE_time`), le signal
  Fan End (`_FE_time`), et la vitesse de rotation (`RPM`). Certains fichiers contiennent aussi
  un signal Base Accelerometer (`_BA_time`).
- Le signal Drive End est notre variable cible : c'est le capteur le plus proche des défauts.
- Le signal est un vecteur de float64, typiquement ~240 000 points soit **~20 secondes** à 12 kHz.
- La **moyenne est proche de zéro** (signal AC centré), ce qui est attendu pour un accéléromètre.

## Section 4 — Comparaison visuelle Normal vs Défauts

In [ ]:
def charger_signal_de(chemin: Path) -> np.ndarray:
    """Charge le signal Drive End d'un fichier .mat CWRU et le retourne en tableau 1D."""
    mat = scipy.io.loadmat(str(chemin))
    de_keys = [k for k in mat.keys() if k.endswith("_DE_time")]
    if not de_keys:
        raise KeyError(f"Aucune clé '_DE_time' dans {chemin.name}")
    return mat[de_keys[0]].flatten()


# Un fichier représentatif par catégorie
# On choisit la sévérité de défaut '007' (0.007 pouce) — la plus légère
fichier_normal = next(DATA_RAW.glob("Normal_*.mat"))
fichier_ball   = next(DATA_RAW.glob("B007_*.mat"))
fichier_ir     = next(DATA_RAW.glob("IR007_*.mat"))
fichier_or     = next(DATA_RAW.glob("OR0076_*.mat"))

signal_normal = charger_signal_de(fichier_normal)
signal_ball   = charger_signal_de(fichier_ball)
signal_ir     = charger_signal_de(fichier_ir)
signal_or     = charger_signal_de(fichier_or)

print("Signaux chargés :")
for label, sig, f in [
    ("Normal", signal_normal, fichier_normal),
    ("Ball",   signal_ball,   fichier_ball),
    ("IR",     signal_ir,     fichier_ir),
    ("OR",     signal_or,     fichier_or),
]:
    print(f"  {label:<8} ← {f.name:<22}  {len(sig):,} points")

In [ ]:
# Fenêtre d'affichage : 0.1 s = 1 200 points
# Cette durée est suffisante pour observer 1 à 3 tours de roulement
NB_POINTS_AFFICHAGE = int(0.1 * SAMPLING_RATE)  # = 1 200
temps_s = np.arange(NB_POINTS_AFFICHAGE) / SAMPLING_RATE

signaux_a_afficher = [
    (signal_normal, "Signal Normal — roulement sain",                         "#4CAF50"),
    (signal_ball,   "Défaut Ball (0.007\" ) — impacts sur la bille roulante",  "#FF9800"),
    (signal_ir,     "Défaut Inner Race (0.007\") — chocs périodiques visibles","#F44336"),
    (signal_or,     "Défaut Outer Race (0.007\") — modulation d'amplitude",    "#2196F3"),
]

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
fig.suptitle(
    "Comparaison des signaux vibratoires CWRU — fenêtre 0–0,1 s",
    fontsize=13, fontweight="bold", y=1.01,
)

for ax, (signal, titre, couleur) in zip(axes, signaux_a_afficher):
    ax.plot(temps_s, signal[:NB_POINTS_AFFICHAGE], color=couleur, linewidth=0.8)
    ax.set_title(titre, fontsize=10, pad=4, loc="left")
    ax.set_ylabel("Amplitude (g)", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

axes[-1].set_xlabel("Temps (s)")
plt.tight_layout()
plt.show()

### Interprétation visuelle

**Signal Normal :** amplitude faible et relativement uniforme, pas de structure impulsionnelle.
C'est le signal de référence que l'autoencoder devra apprendre à reconstruire fidèlement.

**Défaut Ball :** l'impaction de la bille défectueuse se produit à une fréquence caractéristique
(Ball Spin Frequency). Les chocs sont plus diffus que pour les défauts de bague car la bille
roule à la fois sur la bague intérieure et extérieure.

**Défaut Inner Race (IR) :** chocs **périodiques et nets** visibles à l'œil. La bague intérieure
tourne solidaire de l'arbre, donc le défaut passe sous les billes à une fréquence élevée
(BPFI — Ball Pass Frequency Inner Race). L'amplitude des chocs est bien supérieure au Normal.

**Défaut Outer Race (OR) :** la bague extérieure est fixe. Le défaut génère une modulation
d'amplitude caractéristique (BPFO — Ball Pass Frequency Outer Race), souvent à fréquence
plus basse que IR mais avec des chocs d'amplitude élevée.

> **Clé pour l'autoencoder :** entraîné uniquement sur des signaux Normal,
> l'autoencoder aura une **erreur de reconstruction élevée** sur les signaux défectueux
> car ils contiennent des structures impulsionnelles qu'il n'a jamais vues.

## Section 5 — Features statistiques temporelles

In [ ]:
def calculer_features(signal: np.ndarray, label: str) -> dict:
    """
    Calcule les features statistiques temporelles classiques d'un signal vibratoire.

    Parameters
    ----------
    signal : np.ndarray
        Signal 1D d'accélération.
    label : str
        Nom du signal (pour l'index du DataFrame résultat).

    Returns
    -------
    dict avec les features calculées.
    """
    rms = float(np.sqrt(np.mean(signal ** 2)))
    peak = float(np.max(np.abs(signal)))
    return {
        "Type de signal" : label,
        "RMS"            : round(rms, 5),
        "Kurtosis"       : round(float(scipy.stats.kurtosis(signal)), 3),
        "Crest Factor"   : round(peak / rms, 3),
        "Peak-to-Peak"   : round(float(np.max(signal) - np.min(signal)), 5),
        "Ecart-type"     : round(float(np.std(signal)), 5),
    }


rows = [
    calculer_features(signal_normal, "Normal"),
    calculer_features(signal_ball,   "Ball"),
    calculer_features(signal_ir,     "IR"),
    calculer_features(signal_or,     "OR"),
]

df_features = pd.DataFrame(rows).set_index("Type de signal")

print("=== Features statistiques temporelles ===\n")
df_features.style.format("{:.4f}").background_gradient(cmap="YlOrRd", axis=0)

### Interprétation des features

**RMS (Root Mean Square) :** mesure l'énergie globale du signal.
Il augmente avec les défauts car les chocs injectent de l'énergie supplémentaire dans le signal.
C'est une feature robuste mais peu sensible aux défauts naissants.

**Kurtosis :** mesure l'impulsivité de la distribution du signal ("queue de distribution").
Un signal gaussien pur a un kurtosis ≈ 0 (convention Fisher utilisée par scipy).
Les défauts génèrent des chocs → distribution leptokurtique (kurtosis >> 0).
C'est la **feature la plus sensible aux défauts de roulement** dans la littérature.

**Crest Factor (CF) :** rapport entre la valeur de crête et le RMS.
Élevé en début de défaut (chocs intenses, énergie globale encore faible).
Peut diminuer en fin de vie (signal globalement saturé par le bruit de défaut).

**Peak-to-Peak :** amplitude totale du signal. Simple mais sensible au bruit.

**Écart-type :** très proche du RMS pour un signal centré (moyenne ≈ 0),
redondant ici mais utile comme vérification de cohérence.

> Ces features seront calculées sur des **fenêtres glissantes** dans `data_loader.py`
> pour construire le vecteur d'entrée de l'autoencoder.

## Section 6 — Visualisation comparative des features

In [ ]:
features_a_afficher = ["RMS", "Kurtosis", "Crest Factor", "Ecart-type"]

fig, axes = plt.subplots(1, len(features_a_afficher), figsize=(14, 5))
fig.suptitle(
    "Comparaison des features statistiques par catégorie de défaut",
    fontsize=13, fontweight="bold",
)

for ax, feature in zip(axes, features_a_afficher):
    categories_plot = df_features.index.tolist()
    valeurs = df_features[feature].tolist()
    couleurs_plot = [COULEURS[c] for c in categories_plot]

    bars = ax.bar(
        categories_plot, valeurs,
        color=couleurs_plot,
        edgecolor="white", linewidth=1.1,
        width=0.55,
    )
    for bar, val in zip(bars, valeurs):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.03,
            f"{val:.2f}",
            ha="center", fontsize=8,
        )
    ax.set_title(feature, fontweight="bold", pad=8)
    ax.set_ylabel("Valeur", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### Conclusion sur la discriminance des features

D'après les graphiques :

- **Kurtosis** est la feature la plus discriminante : les valeurs pour Ball/IR/OR
  sont nettement supérieures au Normal. Il sera très utile comme feature d'entrée
  et comme critère de détection post-autoencoder.

- **Crest Factor** est aussi discriminant, surtout pour les défauts de bague (IR, OR)
  qui génèrent des chocs très impulsifs.

- **RMS et Écart-type** montrent une séparation plus modeste sur ces fichiers de sévérité
  légère (0.007"). Ils deviendraient plus discriminants avec des défauts plus sévères
  (0.014", 0.021").

> **Pour la suite :** ces features seront calculées sur des fenêtres glissantes
> de 1 024 ou 2 048 points pour construire les séquences d'entraînement de l'autoencoder.

## Section 7 — Conclusion et perspectives

### Ce qu'on a appris dans ce notebook

1. **Structure du dataset** : 40 fichiers `.mat` répartis en 4 catégories (Normal, Ball, IR, OR),
   chaque fichier contenant ~20 s de signal à 12 000 Hz.

2. **Format des données** : les fichiers `.mat` sont lisibles avec `scipy.io.loadmat`;
   la clé Drive End suit le pattern `X<NNN>_DE_time` et retourne un tableau `(N, 1)`
   à aplatir avec `.flatten()`.

3. **Différences visuelles** : les défauts génèrent des chocs périodiques clairement visibles
   sur une fenêtre de 0,1 s, avec une amplitude 2 à 5× supérieure au signal Normal.

4. **Features discriminantes** : le Kurtosis et le Crest Factor séparent bien Normal
   des défauts, même à faible sévérité (0.007"). Le RMS est une baseline utile.

### Prochaines étapes du projet

| Étape | Fichier cible | Description |
|---|---|---|
| 1 | `src/data_loader.py` | Découpe en fenêtres glissantes, normalisation, `torch.Dataset` |
| 2 | `src/config.py` | Centralisation des hyperparamètres (taille fenêtre, stride, batch) |
| 3 | `src/model.py` | Architecture de l'autoencoder (encoder + decoder convolutionnel ou LSTM) |
| 4 | `src/train.py` | Boucle d'entraînement sur les signaux Normal uniquement |
| 5 | `notebooks/02_anomaly_detection.ipynb` | Seuillage de l'erreur de reconstruction, métriques |
